In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc
from sklearn.metrics import roc_auc_score

PROCESSED = Path("../data/processed")
REPORTS = Path("reports")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

In [2]:
pos = pd.read_parquet(PROCESSED / "POS_CASH_balance.parquet")
train_target = pd.read_parquet(
    PROCESSED / "application_train_reduced.parquet",
    columns=["SK_ID_CURR", "TARGET"]
)

print(f"POS_CASH_balance: {pos.shape}")
print(f"Memoria: {pos.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nDtypes:")
print(pos.dtypes)
print(f"\nHead:")
print(pos.head())

POS_CASH_balance: (10001358, 8)
Memoria: 727.1 MB

Dtypes:
SK_ID_PREV                 int32
SK_ID_CURR                 int32
MONTHS_BALANCE              int8
CNT_INSTALMENT           float32
CNT_INSTALMENT_FUTURE    float32
NAME_CONTRACT_STATUS      object
SK_DPD                     int16
SK_DPD_DEF                 int16
dtype: object

Head:
   SK_ID_PREV  SK_ID_CURR  MONTHS_BALANCE  CNT_INSTALMENT  \
0     1803195      182943             -31            48.0   
1     1715348      367990             -33            36.0   
2     1784872      397406             -32            12.0   
3     1903291      269225             -35            48.0   
4     2341044      334279             -35            36.0   

   CNT_INSTALMENT_FUTURE NAME_CONTRACT_STATUS  SK_DPD  SK_DPD_DEF  
0                   45.0               Active       0           0  
1                   35.0               Active       0           0  
2                    9.0               Active       0           0  
3                

In [3]:
n_curr = pos["SK_ID_CURR"].nunique()
n_prev = pos["SK_ID_PREV"].nunique()
train_with_pos = train_target["SK_ID_CURR"].isin(pos["SK_ID_CURR"]).sum()

print(f"SK_ID_CURR únicos:  {n_curr:,}")
print(f"SK_ID_PREV únicos:  {n_prev:,}")
print(f"Clientes train con POS_CASH: {train_with_pos:,} "
      f"({100*train_with_pos/len(train_target):.2f}%)")

print(f"\nMONTHS_BALANCE rango: [{pos['MONTHS_BALANCE'].min()}, {pos['MONTHS_BALANCE'].max()}]")

rows_per_client = pos.groupby("SK_ID_CURR").size()
print(f"\n=== Filas mensuales por cliente ===")
print(rows_per_client.describe().round(1))

SK_ID_CURR únicos:  337,252
SK_ID_PREV únicos:  936,325
Clientes train con POS_CASH: 289,444 (94.12%)

MONTHS_BALANCE rango: [-96, -1]

=== Filas mensuales por cliente ===
count    337252.0
mean         29.7
std          24.5
min           1.0
25%          12.0
50%          22.0
75%          39.0
max         295.0
dtype: float64


In [4]:
print("=== NAME_CONTRACT_STATUS ===")
vc = pos["NAME_CONTRACT_STATUS"].value_counts(dropna=False)
print(pd.DataFrame({"count": vc, "pct": (vc/len(pos)*100).round(3)}))

=== NAME_CONTRACT_STATUS ===
                         count     pct
NAME_CONTRACT_STATUS                  
Active                 9151119  91.499
Completed               744883   7.448
Signed                   87260   0.872
Demand                    7065   0.071
Returned to the store     5461   0.055
Approved                  4917   0.049
Amortized debt             636   0.006
Canceled                    15   0.000
XNA                          2   0.000


In [5]:
print("=== SK_DPD (días de atraso reportado) ===")
print(pos["SK_DPD"].describe().round(1))
print(f"Filas con SK_DPD > 0: {(pos['SK_DPD'] > 0).sum():,} ({(pos['SK_DPD'] > 0).mean()*100:.3f}%)")

print("\n=== SK_DPD_DEF (días de atraso definitivo) ===")
print(pos["SK_DPD_DEF"].describe().round(1))
print(f"Filas con SK_DPD_DEF > 0: {(pos['SK_DPD_DEF'] > 0).sum():,} ({(pos['SK_DPD_DEF'] > 0).mean()*100:.3f}%)")

print("\n=== CNT_INSTALMENT y CNT_INSTALMENT_FUTURE ===")
print(pos[["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]].describe().round(1))

print("\n=== % nulos ===")
print((pos.isna().mean() * 100).round(3).sort_values(ascending=False))

=== SK_DPD (días de atraso reportado) ===
count    10001358.0
mean           11.6
std           132.7
min             0.0
25%             0.0
50%             0.0
75%             0.0
max          4231.0
Name: SK_DPD, dtype: float64
Filas con SK_DPD > 0: 295,227 (2.952%)

=== SK_DPD_DEF (días de atraso definitivo) ===
count    10001358.0
mean            0.7
std            32.8
min             0.0
25%             0.0
50%             0.0
75%             0.0
max          3595.0
Name: SK_DPD_DEF, dtype: float64
Filas con SK_DPD_DEF > 0: 113,969 (1.140%)

=== CNT_INSTALMENT y CNT_INSTALMENT_FUTURE ===
       CNT_INSTALMENT  CNT_INSTALMENT_FUTURE
count       9975287.0              9975271.0
mean             17.1                   10.5
std              11.5                   11.0
min               1.0                    0.0
25%              10.0                    3.0
50%              12.0                    7.0
75%              24.0                   14.0
max              92.0                 

In [6]:
# Indicadores binarios derivados
pos["POS_HAS_DPD"] = (pos["SK_DPD"] > 0).astype("int8")
pos["POS_HAS_DPD_DEF"] = (pos["SK_DPD_DEF"] > 0).astype("int8")
pos["POS_IS_COMPLETED"] = (pos["NAME_CONTRACT_STATUS"] == "Completed").astype("int8")
pos["POS_IS_ACTIVE"] = (pos["NAME_CONTRACT_STATUS"] == "Active").astype("int8")

agg_dict = {
    "MONTHS_BALANCE": ["count", "min", "max", "mean"],
    "SK_DPD": ["max", "mean", "sum"],
    "SK_DPD_DEF": ["max", "mean", "sum"],
    "CNT_INSTALMENT": ["max", "mean"],
    "CNT_INSTALMENT_FUTURE": ["max", "mean"],
    "POS_HAS_DPD": ["max", "sum", "mean"],
    "POS_HAS_DPD_DEF": ["max", "sum", "mean"],
    "POS_IS_COMPLETED": ["max", "mean"],
    "POS_IS_ACTIVE": ["max", "mean"],
}

pos_agg = pos.groupby("SK_ID_CURR").agg(agg_dict)
pos_agg.columns = ["POS_" + "_".join(c).upper() for c in pos_agg.columns]
pos_agg = pos_agg.reset_index()

# Conteo de SK_ID_PREV distintos por cliente
n_prevs = pos.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().rename("POS_N_PREV_LOANS")
pos_agg = pos_agg.merge(n_prevs, on="SK_ID_CURR")

print(f"Shape: {pos_agg.shape}")
print(pos_agg.head(3))

Shape: (337252, 26)
   SK_ID_CURR  POS_MONTHS_BALANCE_COUNT  POS_MONTHS_BALANCE_MIN  \
0      100001                         9                     -96   
1      100002                        19                     -19   
2      100003                        28                     -77   

   POS_MONTHS_BALANCE_MAX  POS_MONTHS_BALANCE_MEAN  POS_SK_DPD_MAX  \
0                     -53               -72.555556               7   
1                      -1               -10.000000               0   
2                     -18               -43.785714               0   

   POS_SK_DPD_MEAN  POS_SK_DPD_SUM  POS_SK_DPD_DEF_MAX  POS_SK_DPD_DEF_MEAN  \
0         0.777778               7                   7             0.777778   
1         0.000000               0                   0             0.000000   
2         0.000000               0                   0             0.000000   

   POS_SK_DPD_DEF_SUM  POS_CNT_INSTALMENT_MAX  POS_CNT_INSTALMENT_MEAN  \
0                   7                  

In [7]:
pos_target = pos_agg.merge(train_target, on="SK_ID_CURR", how="inner")
print(f"Clientes con POS_CASH + target: {pos_target.shape}")

pos_features = [c for c in pos_target.columns if c.startswith("POS_")]
auc_list = []

for col in pos_features:
    m = pos_target[col].notna()
    if m.sum() == 0 or pos_target.loc[m, col].nunique() < 2:
        continue
        
    auc_raw = roc_auc_score(pos_target.loc[m, "TARGET"], pos_target.loc[m, col])
    auc_list.append({
        "feature": col,
        "auc": round(max(auc_raw, 1 - auc_raw), 4),
        "direction": "↓ menos default" if auc_raw < 0.5 else "↑ más default",
        "pct_null": round(100 * pos_target[col].isna().mean(), 2),
    })

auc_pos = pd.DataFrame(auc_list).sort_values("auc", ascending=False)
print(f"\n=== Top features de POS_CASH por AUC ===")
print(auc_pos.to_string(index=False))
auc_pos.to_csv(REPORTS / "auc_rank_pos_cash.csv", index=False)

Clientes con POS_CASH + target: (289444, 27)

=== Top features de POS_CASH por AUC ===
                       feature    auc       direction  pct_null
        POS_MONTHS_BALANCE_MIN 0.5584   ↑ más default      0.00
              POS_N_PREV_LOANS 0.5491 ↓ menos default      0.00
      POS_MONTHS_BALANCE_COUNT 0.5456 ↓ menos default      0.00
       POS_MONTHS_BALANCE_MEAN 0.5437   ↑ más default      0.00
           POS_SK_DPD_DEF_MEAN 0.5261   ↑ más default      0.00
          POS_POS_HAS_DPD_MEAN 0.5260   ↑ más default      0.00
      POS_POS_HAS_DPD_DEF_MEAN 0.5260   ↑ más default      0.00
               POS_SK_DPD_MEAN 0.5254   ↑ más default      0.00
            POS_SK_DPD_DEF_MAX 0.5253   ↑ más default      0.00
            POS_SK_DPD_DEF_SUM 0.5252   ↑ más default      0.00
       POS_POS_HAS_DPD_DEF_SUM 0.5246   ↑ más default      0.00
       POS_POS_HAS_DPD_DEF_MAX 0.5242   ↑ más default      0.00
                POS_SK_DPD_MAX 0.5241   ↑ más default      0.00
                P

In [8]:
pos_agg.to_parquet(PROCESSED / "pos_cash_aggregated.parquet", index=False)
mem = pos_agg.memory_usage(deep=True).sum() / 1024**2
print(f"Guardado: pos_cash_aggregated.parquet")
print(f"Shape: {pos_agg.shape} | Memoria: {mem:.1f} MB")

del pos, pos_target
gc.collect()

Guardado: pos_cash_aggregated.parquet
Shape: (337252, 26) | Memoria: 43.1 MB


10